# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [ ]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [ ]:
MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

# TODO: replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = "student_your_username"
MLFLOW_PASSWORD = "your_mlflow_password"
EXPERIMENT_NAME = "qbc12_hw02_student_your_username"

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [ ]:
DATASET_VERSION = "v1_audit_cleaned"

FEATURE_DIR = Path("data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# TODO: load the dataset.
# Prefer Parquet if it exists, otherwise use CSV.
raise NotImplementedError("Load feature_df from parquet_path or csv_path.")

# TODO: load metadata if metadata_path exists.
metadata = {}

print(feature_df.shape)
feature_df.head()

## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [ ]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

# TODO: check that TARGET_COL exists.
# TODO: create y.
# TODO: create clean feature list by excluding FORBIDDEN_MODEL_COLUMNS.
# TODO: create X_clean.

raise NotImplementedError("Create y, clean_feature_cols, and X_clean.")

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [ ]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# TODO: create leaky_feature_cols.
# It should include the clean features plus LEAKAGE_COLUMN.
# It must still exclude the target itself.

raise NotImplementedError("Create leaky_feature_cols and X_leaky.")

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [ ]:
# TODO: split X_clean and y.
# Use test_size=0.20, random_state=42, stratify=y.

raise NotImplementedError("Create X_train, X_test, y_train, y_test.")

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [ ]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# TODO: identify numeric_cols and categorical_cols from X_clean.
# Hint: numeric columns usually have dtype int/float.
# Everything else can be treated as categorical.

raise NotImplementedError("Create numeric_cols and categorical_cols.")

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [ ]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    # TODO:
    # 1. get positive scores
    # 2. convert scores to predictions using threshold
    # 3. calculate accuracy, precision, recall, f1, roc_auc
    # 4. return metrics dict, y_pred, y_score

    raise NotImplementedError("Complete evaluate_binary_classifier.")

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [ ]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    # TODO:
    # 1. create a run-specific artifact folder
    # 2. save confusion_matrix.png
    # 3. save classification_report.json
    # 4. save feature_columns.json
    # 5. save dataset_metadata_snapshot.json

    raise NotImplementedError("Complete save_run_artifacts.")

## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [ ]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    # TODO: implement this function.
    # Required MLflow calls:
    # - mlflow.start_run(run_name=run_name)
    # - mlflow.log_params(...)
    # - mlflow.log_metrics(...)
    # - mlflow.set_tags(...)
    # - mlflow.log_artifacts(...)
    # - mlflow.sklearn.log_model(...)

    raise NotImplementedError("Complete run_mlflow_experiment.")

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [ ]:
# TODO:
# 1. split X_leaky and y using the same stratified split settings
# 2. build a LogisticRegression pipeline
# 3. log the run to MLflow

raise NotImplementedError("Run and log v0_leaky_logistic_regression.")

## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [ ]:
# TODO: build and log dummy baseline.

raise NotImplementedError("Run and log v1_dummy_baseline.")

## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [ ]:
# TODO: build and log clean LogisticRegression.

raise NotImplementedError("Run and log v2_clean_logistic_regression.")

## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [ ]:
# TODO: build and log class-weighted LogisticRegression.

raise NotImplementedError("Run and log v3_balanced_logistic_regression.")

## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [ ]:
# TODO: log threshold-tuning runs.

raise NotImplementedError("Run and log threshold tuning experiments.")

## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [ ]:
# TODO: build and log RandomForestClassifier.

raise NotImplementedError("Run and log v5_random_forest.")

## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# TODO: retrieve MLflow runs for this experiment and create a comparison table.
raise NotImplementedError("Create runs comparison table.")

comparison_df

## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [ ]:
# TODO: set BEST_RUN_ID to the selected clean run ID.
BEST_RUN_ID = None

if BEST_RUN_ID is None:
    raise ValueError("Set BEST_RUN_ID to your selected clean MLflow run ID.")

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)

## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [ ]:
# TODO: replace this text.
final_explanation = """
Write your final explanation here.
"""

print(final_explanation)